# Zusätzliche Downstream-Aufgaben und Latentraum-Plots

Nur zwei Schritte: **native AUTOENCODIX-LogisticRegression-Auswertung** auf gespeicherten Posteriors und **Latentraum-Visualisierung**. Kein VAE-Training, keine eigene AUC-Implementierung, keine Regressionsaufgaben.

Train/Valid/Test bleiben unverändert. `z_redrawn` ist ein reproduzierbar neu gezogenes z aus μ/logvar, nicht die ursprüngliche AUTOENCODIX-Ziehung. Für μ `REPRESENTATION='mu'` einstellen. Die Probe erhält die vollständigen Latentdimensionen, nicht die UMAP-Koordinaten.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from umap import UMAP
from IPython.display import display
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'beta_vae_experiments/data_pipeline.py').is_file()
            or (p / 'vae-beta-control/beta_vae_experiments/data_pipeline.py').is_file())
if not (ROOT / 'beta_vae_experiments/data_pipeline.py').is_file(): ROOT = ROOT / 'vae-beta-control'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from autoencodix.evaluate._general_evaluator import GeneralEvaluator
from autoencodix.visualize import GeneralVisualizer
SWEEP = ROOT / 'beta_vae_experiments/new_results_2026-09-29/biology_breakpoints_2026_09_29'
OUT = ROOT / 'beta_vae_experiments/results_26_09/additional_downstream_tasks/native'
OUT.mkdir(parents=True, exist_ok=True)
CLIN = pd.read_parquet(ROOT.parent / 'autoencodix_package/data/raw/OntixTutorialData/combined_clin_formatted.parquet')
CLIN.index = CLIN.index.astype(str)
assert CLIN.index.is_unique
RUNS = pd.read_csv(SWEEP / 'experiment_metrics.csv')
RUNS = RUNS[RUNS.status.eq('ok')].copy()
RUNS['beta_per_dim'] = (RUNS.beta / RUNS.latent_dim).round(10)
REPRESENTATION = 'z_redrawn'  # alternativ: 'mu'
RANDOM_STATE = 31
MIN_TRAIN_CLASS = 20
TASKS = {
    'cancer_type': ('CANCER_TYPE_ACRONYM', None),
    'cancer_groups': ('CANCER_TYPE', None),
    'brca_subtypes': ('SUBTYPE', ['BRCA']),
    'ucec_subtypes': ('SUBTYPE', ['UCEC']),
    'detailed_histology': ('CANCER_TYPE_DETAILED', None),
    'stage_I_IV': ('AJCC_PATHOLOGIC_TUMOR_STAGE_SHORT', None),
    'sex_all': ('SEX', None),
    'sex_mixed': ('SEX', ['COAD', 'LUAD', 'LUSC', 'READ']),
}
print('Quelle:', SWEEP, '| Modelle:', len(RUNS))


## Daten vorbereiten

Unbekannte Labels ausschließen; Klassen mit weniger als 20 Trainproben werden explizit protokolliert. Die Auswahl erfolgt ausschließlich auf Train. Fehlende Klassen in einem Evaluationssplit ergeben eine undefinierte macro-AUC, keine Ersatzwerte. Die native Funktion bekommt für alle auswertbaren Splits dieselbe trainierte Probe.


In [ ]:
def latent_frames(run_name):
    frames = {}
    rng = np.random.default_rng(RANDOM_STATE)
    for split in ['train', 'valid', 'test']:
        path = SWEEP / 'run_analysis' / run_name / 'posteriors' / f'posterior_{split}.npz'
        with np.load(path, allow_pickle=False) as a:
            mu, lv, ids = a['mu'].astype(float), a['logvar'].astype(float), a['sample_ids'].astype(str)
        if mu.shape != lv.shape or len(ids) != len(mu) or len(set(ids)) != len(ids):
            raise ValueError('Inkonsistente Posterior-Shapes/IDs')
        if not pd.Index(ids).isin(CLIN.index).all(): raise ValueError('Fehlende Klinik-IDs')
        if not np.isfinite(mu).all() or not np.isfinite(lv).all(): raise ValueError('Nichtendliche Posteriors')
        if REPRESENTATION not in ['mu', 'z_redrawn']: raise ValueError('Unbekannte Repräsentation')
        values = mu if REPRESENTATION == 'mu' else mu + np.exp(.5 * lv) * rng.standard_normal(mu.shape)
        frames[split] = pd.DataFrame(values, index=ids)
    ids = [set(x.index) for x in frames.values()]
    if any(ids[i] & ids[j] for i in range(3) for j in range(i)): raise ValueError('Überlappende Datensplits')
    if len({x.shape[1] for x in frames.values()}) != 1: raise ValueError('Unterschiedliche Latentdimensionen')
    return frames


def task_frames(frames, task_name):
    target, cohort = TASKS[task_name]
    selected, audit = {}, []
    for split, frame in frames.items():
        ann = CLIN.reindex(frame.index)
        label = ann[target].astype('string').str.strip()
        keep = label.notna() & ~label.str.lower().isin(['unknown', '', 'nan', 'none'])
        if cohort: keep &= ann.CANCER_TYPE_ACRONYM.isin(cohort)
        selected[split] = frame.loc[keep]
    train_labels = CLIN.loc[selected['train'].index, target]
    counts = train_labels.value_counts()
    allowed = counts[counts >= MIN_TRAIN_CLASS].index
    for split, frame in selected.items():
        labels = CLIN.loc[frame.index, target]
        selected[split] = frame.loc[labels.isin(allowed)]
        audit.append({'task': task_name, 'split': split, 'n_samples': len(selected[split]),
                      'excluded_train_classes': ', '.join(counts[counts < MIN_TRAIN_CLASS].index.astype(str)),
                      'class_counts': json.dumps(CLIN.loc[selected[split].index, target].value_counts().to_dict())})
    return selected, pd.DataFrame(audit)


## Native logistische Regression

Der entscheidende Aufruf ist `GeneralEvaluator._single_ml_presplit(..., sklearn_ml=LogisticRegression(max_iter=1000), metric='roc_auc_ovo', ml_type='classification')` — derselbe native Weg wie bisher. Keine separate Berechnung mit `roc_auc_score`.

Standardmäßig ein vorhandenes Modell als Funktionsprüfung. Für den gesamten Sweep `RUN_SELECTION = RUNS` setzen. Dies trainiert nur Probes, kein VAE.


In [ ]:
RUN_SELECTION = RUNS[(RUNS.latent_dim == 8) & (RUNS.seed == 13) & (RUNS.beta == 0)]
results, inventories = [], []
for run in RUN_SELECTION.itertuples():
    frames = latent_frames(run.run_name)
    for name, (target, cohort) in TASKS.items():
        selected, audit = task_frames(frames, name)
        inventories.append(audit.assign(run_name=run.run_name))
        classes = set(CLIN.loc[selected['train'].index, target])
        if len(classes) < 2:
            print(name, ': weniger als zwei Trainklassen, übersprungen')
            continue
        eligible = {'train': selected['train']}
        missing = []
        for split in ['valid', 'test']:
            if set(CLIN.loc[selected[split].index, target]) == classes:
                eligible[split] = selected[split]
            else:
                missing.append({'score_split': split, 'CLINIC_PARAM': target, 'metric': 'roc_auc_ovo',
                                'value': np.nan, 'status': 'missing_evaluation_class'})
        sample_split = pd.DataFrame([{'SAMPLE_ID': sid, 'SPLIT': split}
                                    for split, frame in eligible.items() for sid in frame.index])
        scores = GeneralEvaluator._single_ml_presplit(
            sample_split=sample_split, df=pd.concat(eligible.values()), clin_data=CLIN,
            task_param=target, sklearn_ml=LogisticRegression(max_iter=1000),
            metric='roc_auc_ovo', ml_type='classification', top_k_classes=None)
        scores['status'] = 'ok'
        if missing: scores = pd.concat([scores, pd.DataFrame(missing)], ignore_index=True)
        results.append(scores.assign(task=name, run_name=run.run_name, latent_dim=run.latent_dim,
                                     seed=run.seed, beta=run.beta, beta_per_dim=run.beta_per_dim,
                                     representation=REPRESENTATION))
SCORES = pd.concat(results, ignore_index=True)
SCORES.to_parquet(OUT / f'downstream_{REPRESENTATION}.parquet', index=False)
INVENTORY = pd.concat(inventories, ignore_index=True)
INVENTORY.to_parquet(OUT / 'label_inventory.parquet', index=False)
display(SCORES.pivot_table(index='task', columns='score_split', values='value', aggfunc='mean'))
display(INVENTORY)


## Latentraum-Diagramme wie in der Präsentation

Vorhandene Folien-Koordinaten aus `plots/<run>/2D_embedding_z_all.csv` wiederverwenden: ein damaliger z-Zug, UMAP auf **Train + Valid + Test gemeinsam**. Keine neue UMAP-Anpassung. Beide Panels eines Diagramms zeigen exakt dasselbe Modell und dieselben Koordinaten. AUTOENCODIX zeichnet die Punkte und 2D-Gruppenmittel; die zusätzliche Formatierung betrifft nur Farben, Sterne, Titel und Legende.

Geschlecht: **alle Krebsarten** und **gemischte Krebsarten** getrennt, wie bei der nativen LogisticRegression oben. Bei den ausgewählten Typen bleiben andere Krebsarten grau, unbekanntes Geschlecht mittelgrau. Sterne zeigen den Mittelwert **pro Krebsart und Geschlecht in 2D**, wie in den Folien; sie sind keine Posterior-Mittelwerte.

Die alte Cache-z-Ziehung unterscheidet sich von der neuen Probe-z-Ziehung. Deshalb keine Notebook-AUC an den Cache-Plots anzeigen. Verschiedene β-Modelle haben unabhängig angepasste UMAP-Achsen. Geschlechtsleistung über alle Krebsarten kann Krebsartenunterschiede ausnutzen; die Teilkohorte ist eine ergänzende Kontrolle, keine vollständige Adjustierung für Krebsart.


In [ ]:
import io, base64
from IPython.display import HTML
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba

PLOT_D = 8
PLOT_SEED = 13
PLOT_BETAS = [0., 5e-5, 1e-3]
PLOT_TASK_PAIRS = [('sex_all', 'sex_mixed'),
                   ('cancer_type', 'brca_subtypes'),
                   ('cancer_type', 'ucec_subtypes')]


def cached_slide_embedding(run):
    path = SWEEP / 'plots' / run.run_name / '2D_embedding_z_all.csv'
    if not path.is_file(): raise FileNotFoundError('Folien-Koordinaten fehlen: ' + str(path))
    embedding = pd.read_csv(path, index_col=0)
    embedding.index = embedding.index.astype(str)
    frames = latent_frames(run.run_name)
    expected = pd.Index(np.concatenate([frames[s].index.to_numpy() for s in ['train', 'valid', 'test']]))
    if not embedding.index.is_unique or set(embedding.index) != set(expected):
        raise ValueError('Folien-Koordinaten und Posterior-IDs stimmen nicht überein')
    if embedding.shape[1] != 2 or not np.isfinite(embedding.to_numpy()).all():
        raise ValueError('Ungültige 2D-Koordinaten')
    embedding = embedding.reindex(expected)
    embedding.columns = [0, 1]
    return embedding, frames


def slide_panel(embedding, frames, task_name, beta_per_dim):
    target, cohort = TASKS[task_name]
    ann = CLIN.reindex(embedding.index)
    selected, _ = task_frames(frames, task_name)
    eligible = set().union(*(set(f.index) for f in selected.values()))
    labels = ann[target].fillna('unknown').astype(str)
    in_cohort = ann.CANCER_TYPE_ACRONYM.isin(cohort) if cohort else pd.Series(True, index=ann.index)
    plotted = labels.where(embedding.index.isin(eligible), 'other')
    unknown = in_cohort & labels.str.lower().isin(['unknown', '', 'nan', 'none'])
    plotted = plotted.mask(unknown, 'unknown')
    # Native AUTOENCODIX renderer; recolour its collections for deck styling.
    fig = GeneralVisualizer._plot_2D(
        embedding=embedding.reset_index(drop=True), labels=plotted.tolist(),
        param=task_name, layer='', center=True, figsize=(9, 7))
    ax = fig.axes[0]
    categories = sorted(set(plotted) - {'other', 'unknown'})
    palette = {c: plt.get_cmap('tab20' if len(categories)>10 else 'tab10')(i)
               for i, c in enumerate(categories)}
    if target == 'SEX': palette.update({'Female': '#00A6A6', 'Male': '#1A1A1A'})
    palette.update({'other': '#DCE3EC', 'unknown': '#9AA6B5'})
    points = ax.collections[0]
    points.set_facecolors([to_rgba(palette[v]) for v in plotted])
    points.set_edgecolors('none'); points.set_sizes([7]); points.set_alpha(.75)
    # Replace native class stars with the deck's grouping convention.
    if len(ax.collections)>1: ax.collections[1].remove()
    marker_frame = embedding.copy(); marker_frame['label'] = plotted.to_numpy()
    marker_frame['cancer'] = ann.CANCER_TYPE_ACRONYM.astype(str).to_numpy()
    marked = marker_frame[~marker_frame.label.isin(['other', 'unknown'])]
    grouping = ['cancer', 'label'] if target == 'SEX' else ['label']
    for group, values in marked.groupby(grouping):
        label = group[-1] if isinstance(group, tuple) else group
        ax.scatter(values[0].mean(), values[1].mean(), marker='*', s=450,
                   color=palette[label], edgecolor='black', linewidth=1.2, zorder=5)
    title = {'sex_all':'SEX · ALL CANCER TYPES', 'sex_mixed':'SEX · COAD / LUAD / LUSC / READ',
             'cancer_type':'CANCER TYPE', 'brca_subtypes':'BRCA · MOLECULAR SUBTYPES',
             'ucec_subtypes':'UCEC · MOLECULAR SUBTYPES'}.get(task_name, task_name.upper())
    ax.set_title(title, color='white', fontsize=14, fontweight='bold', pad=16,
                 bbox={'boxstyle':'round,pad=.35', 'facecolor':'#00A6A6' if target=='SEX' else '#2F6DB5', 'edgecolor':'none'})
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(''); ax.set_ylabel(''); ax.grid(False)
    for spine in ax.spines.values(): spine.set_color('#DCE3EC')
    handles = [Line2D([], [], marker='o', linestyle='', color=palette[c],
                      label={'other':'outside task cohort / excluded class', 'unknown':'label unknown'}.get(c,c))
               for c in categories + [c for c in ['unknown','other'] if c in set(plotted)]]
    handles.append(Line2D([],[],marker='*',linestyle='',markerfacecolor='white',markeredgecolor='black',markersize=14,
                          label='2D mean per cancer type and sex' if target=='SEX' else '2D class mean'))
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5,-.02), ncol=min(4,len(handles)), frameon=False, fontsize=9)
    fig.tight_layout()
    stem = f'slide_latent_{task_name}_d{PLOT_D}_seed{PLOT_SEED}_beta{beta_per_dim:g}'
    fig.savefig(OUT / f'{stem}.png', dpi=180, bbox_inches='tight', facecolor='white')
    stream = io.BytesIO(); fig.savefig(stream, format='png', dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return base64.b64encode(stream.getvalue()).decode()


for beta_per_dim in PLOT_BETAS:
    choice = RUNS[(RUNS.latent_dim==PLOT_D) & (RUNS.seed==PLOT_SEED)
                  & np.isclose(RUNS.beta_per_dim,beta_per_dim,rtol=0,atol=1e-10)]
    if len(choice)!=1: raise ValueError('Plot-Modell fehlt oder ist nicht eindeutig')
    run = choice.iloc[0]
    embedding, frames = cached_slide_embedding(run)
    for left, right in PLOT_TASK_PAIRS:
        lpng = slide_panel(embedding,frames,left,beta_per_dim)
        rpng = slide_panel(embedding,frames,right,beta_per_dim)
        html = f'<h3>d={PLOT_D}, seed={PLOT_SEED}, β/d={beta_per_dim:g} · cached z · all splits</h3><div style="display:flex;gap:12px;align-items:flex-start"><img style="width:49%" src="data:image/png;base64,{lpng}"><img style="width:49%" src="data:image/png;base64,{rpng}"></div>'
        (OUT / f'paired_{left}_{right}_d{PLOT_D}_seed{PLOT_SEED}_beta{beta_per_dim:g}.html').write_text(html)
        display(HTML(html))


## Frequenzbänder des Referenzgraphen und der Downstream-Aufgaben

Derselbe **k=20-Referenzgraph auf den 707 ursprünglichen Testproben** wie im anderen Notebook, dieselbe normalisierte Laplace-Basis und disjunkte B0–B6-Bänder. Keine neue Graphdefinition. Die vorhandenen UMAP-Plots zeigen dagegen alle 3529 Train-/Valid-/Testproben; UMAP-Koordinaten werden nicht für die Frequenzanalyse benutzt.

Drei Ansichten: Eigenwertspektrum mit Bandgrenzen; Labelenergie der zusätzlichen Aufgaben je Band; Energie der Encoder-Mittelwerte μ je Dimension und Band bei ausgewählten β-Werten. Labels sind hier eine deskriptive Kalibrierung, kein Trainingssignal und keine Vorhersagegarantie.


In [ ]:
from beta_vae_experiments.metrics.graph_signals import graph_spectrum as GSP
from beta_vae_experiments.metrics.graph_signals.dimension_attribution import reference_bands, fingerprints, align_rows
from beta_vae_experiments.result_analysis.utils import DEFAULT_X, posterior
SPECTRAL_DIR = ROOT / 'beta_vae_experiments/results_26_09/spectral_dimension_analysis'
BASIS_PATH = SPECTRAL_DIR / 'reference_basis.npz'
if not BASIS_PATH.is_file(): raise FileNotFoundError('Bestehende Referenzbasis fehlt: ' + str(BASIS_PATH))
with np.load(BASIS_PATH, allow_pickle=False) as a:
    U, LAM, GRAPH_IDS, GRAPH_INDICES = a['U'], a['lam'], a['sample_ids'].astype(str), a['indices']
with np.load(DEFAULT_X, allow_pickle=False) as a:
    GRAPH_X = a['x_test'][GRAPH_INDICES].astype(float)
    if not np.array_equal(a['ids_test'][GRAPH_INDICES].astype(str), GRAPH_IDS): raise ValueError('Graph-Sample-IDs stimmen nicht überein')
W, L = GSP.build_graph(GRAPH_X, k=20, laplacian='normalized', weights='binary')
if not np.allclose(L @ U, U * LAM, atol=1e-7): raise ValueError('Basis stimmt nicht mit dem Eingabegraphen überein')
if not np.allclose(U.T @ U, np.eye(U.shape[1]), atol=1e-8): raise ValueError('Basis ist nicht orthonormal')
BANDS, BAND_NAMES = reference_bands(LAM)
BAND_TABLE = pd.DataFrame([{'band': name, 'n_modes': len(idx), 'lambda_min': LAM[idx].min(), 'lambda_max': LAM[idx].max()}
                           for name, idx in zip(BAND_NAMES, BANDS)])
display(BAND_TABLE)
BAND_TABLE.to_parquet(OUT / 'frequency_bands.parquet', index=False)
fig, ax = plt.subplots(figsize=(11, 4))
for b, (name, idx) in enumerate(zip(BAND_NAMES, BANDS)):
    shown = idx[idx < 60]
    ax.scatter(shown, LAM[shown], s=35, color=plt.get_cmap('viridis')(b / max(1, len(BANDS)-1)),
               label=f'{name}: {len(idx)} modes, λ≤{LAM[idx].max():.3g}')
ax.set_xlabel('Graph-Laplace mode index'); ax.set_ylabel('λ (normalized Laplacian)')
ax.set_title('Same TCGA input graph · first 60 modes · k=20'); ax.legend(fontsize=9, ncol=2)
fig.tight_layout(); fig.savefig(OUT / 'frequency_band_spectrum.png', dpi=160); display(fig); plt.close(fig)


### In welchen Bändern liegen die Aufgaben?

One-vs-rest-Labelsignale pro Klasse, zentriert **innerhalb der jeweiligen bekannten Aufgabenkohorte**, außerhalb auf null. Dadurch wird „BRCA versus übrige Krebsarten“ nicht als BRCA-Subtyp-Signal mitgezählt. Unbekannte Labels sind keine Klasse. Gezeigt wird der Mittelwert der Energieanteile über auswertbare Klassen.

`sex_all` und `sex_mixed` verwenden die Geschlechtslabels ihrer jeweiligen Kohorte. Zusätzlich zeigt `sex_mixed_within_type` das nach Krebsart zentrierte Geschlechtssignal. Die eigentliche logistische Regression oben bleibt unverändert. Bandgrößen sind sehr ungleich: B6 enthält 680 Moden, daher zusätzlich die Anreicherung gegenüber isotroper Rauschenergie beachten. Hohe Frequenz ist nicht automatisch Rauschen.

Die Rauschreferenz berücksichtigt dieselbe bekannte Kohorte und Zentrierung. Für das adjustierte Geschlecht wird innerhalb jeder Krebsart zentriert. Log₂-Anreicherung wird für die Darstellung bei 10⁻⁶ begrenzt; undefinierte Werte bleiben leer.


In [ ]:
def centred_cohort_noise_fraction(masks):
    # Exact expectation for isotropic noise under the SAME cohort centring;
    # trace(U_B.T @ H_cohort @ U_B)/(number of nonconstant noise directions).
    expected = np.zeros(len(BANDS)); rank = 0
    for mask in masks:
        n_known = int(np.sum(mask))
        if n_known < 2: continue
        block = U[mask]; rank += n_known - 1
        for b, idx in enumerate(BANDS):
            expected[b] += np.sum(block[:,idx]**2) - np.sum(block[:,idx].sum(0)**2)/n_known
    return np.maximum(expected,0)/rank if rank else np.full(len(BANDS),np.nan)

TASK_BAND_ROWS = []
reference_run = RUNS[(RUNS.latent_dim==PLOT_D) & (RUNS.seed==PLOT_SEED) & RUNS.beta.eq(0)].iloc[0]
reference_frames = latent_frames(reference_run.run_name)
for task_name, (target, cohort) in TASKS.items():
    eligible, audit = task_frames(reference_frames, task_name)
    mask = pd.Index(GRAPH_IDS).isin(eligible['test'].index)
    noise = centred_cohort_noise_fraction([mask])
    labels = CLIN[target].reindex(GRAPH_IDS).astype(str).to_numpy()
    classes = sorted(CLIN.loc[eligible['train'].index, target].astype(str).unique())
    signals = np.zeros((len(GRAPH_IDS), len(classes)))
    for j, label in enumerate(classes):
        if mask.any(): signals[mask, j] = (labels[mask]==label).astype(float) - (labels[mask]==label).mean()
    coefficients = GSP.gft(U, signals)
    energy = GSP.band_energy(coefficients, BANDS, normalize=False)
    total = ((signals - signals.mean(0))**2).sum(0)
    fraction = np.divide(energy, total[:, None], out=np.full_like(energy, np.nan), where=total[:, None]>1e-24)
    for j, label in enumerate(classes):
        for b, band in enumerate(BAND_NAMES):
            TASK_BAND_ROWS.append({'task':task_name, 'class':label, 'band':band,
                                   'energy_fraction':fraction[j,b], 'noise_expected_fraction':noise[b], 'energy_abs':energy[j,b],
                                   'n_known_cohort':int(mask.sum()), 'n_positive':int((mask & (labels==label)).sum())})
# Explicit cancer-type-adjusted sex signal in the four mixed cancer types.
ann = CLIN.reindex(GRAPH_IDS)
known = ann.SEX.isin(['Female','Male']) & ann.CANCER_TYPE_ACRONYM.isin(TASKS['sex_mixed'][1])
male = ann.SEX.eq('Male').astype(float)
within = np.zeros(len(GRAPH_IDS))
for cancer in TASKS['sex_mixed'][1]:
    mask = (known & ann.CANCER_TYPE_ACRONYM.eq(cancer)).to_numpy()
    if mask.any(): within[mask] = male.to_numpy()[mask] - male.to_numpy()[mask].mean()
c = GSP.gft(U, within); e = GSP.band_energy(c, BANDS, normalize=False)[0]
f = e / np.sum((within-within.mean())**2) if np.var(within)>0 else np.full(len(e),np.nan)
noise = centred_cohort_noise_fraction([(known & ann.CANCER_TYPE_ACRONYM.eq(cancer)).to_numpy() for cancer in TASKS['sex_mixed'][1]])
for b, band in enumerate(BAND_NAMES):
    TASK_BAND_ROWS.append({'task':'sex_mixed_within_type','class':'male minus within-type prevalence','band':band,
                           'energy_fraction':f[b],'noise_expected_fraction':noise[b],'energy_abs':e[b],'n_known_cohort':int(known.sum()),'n_positive':int((known & ann.SEX.eq('Male')).sum())})
TASK_BANDS = pd.DataFrame(TASK_BAND_ROWS)
TASK_BANDS.to_parquet(OUT / 'task_frequency_profiles.parquet', index=False)
TASK_MEAN = TASK_BANDS.pivot_table(index='task', columns='band', values='energy_fraction', aggfunc='mean').reindex(columns=BAND_NAMES)
EXPECTED_NOISE = TASK_BANDS.pivot_table(index='task',columns='band',values='noise_expected_fraction',aggfunc='first').reindex(index=TASK_MEAN.index,columns=BAND_NAMES)
ENRICHMENT = TASK_MEAN / EXPECTED_NOISE.where(EXPECTED_NOISE>1e-24)
display(TASK_MEAN.round(3))
display(ENRICHMENT.round(2))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, matrix, title in [(axes[0], TASK_MEAN, 'Task label energy fraction'),
                          (axes[1], np.log2(ENRICHMENT.clip(lower=1e-6)), 'log₂ enrichment vs. isotropic noise')]:
    im = ax.imshow(matrix, aspect='auto', cmap='viridis')
    ax.set_yticks(range(len(matrix)), matrix.index); ax.set_xticks(range(len(BAND_NAMES)), BAND_NAMES)
    ax.set_xlabel('Disjoint frequency band · low → high'); ax.set_title(title)
    fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(OUT / 'downstream_task_frequency_bands.png', dpi=170, bbox_inches='tight'); display(fig); plt.close(fig)


### Welche Bänder tragen die latenten Dimensionen bei β?

Hier ausschließlich **μ auf den 707 Referenz-Testproben**, nicht das z der UMAP-Folien. Arithmetische Sample-Zentrierung, gesamter nicht erfasster Energierest separat; konstantes μ hat undefinierte Anteile. Dieselbe Implementierung wie im Forschungsabschnitt des anderen Notebooks. Neuronnummern gehören zum jeweiligen unabhängig trainierten Modell; gleiche Nummer heißt nicht gleiche biologische Bedeutung.


In [ ]:
MU_FREQUENCY_ROWS = []
fig, axes = plt.subplots(1, len(PLOT_BETAS), figsize=(5*len(PLOT_BETAS), 4.8), squeeze=False)
for ax, beta_per_dim in zip(axes[0], PLOT_BETAS):
    choice = RUNS[(RUNS.latent_dim==PLOT_D) & (RUNS.seed==PLOT_SEED)
                  & np.isclose(RUNS.beta_per_dim,beta_per_dim,rtol=0,atol=1e-10)]
    if len(choice)!=1: raise ValueError('Frequenzmodell fehlt oder ist nicht eindeutig')
    run = choice.iloc[0]
    mu, lv, ids = posterior(SWEEP / 'run_analysis' / run.run_name / 'posteriors', 'test')
    mu, lv = align_rows(mu,ids,GRAPH_IDS), align_rows(lv,ids,GRAPH_IDS)
    profile = fingerprints(U,LAM,mu,lv,BANDS,validate_basis=False)
    for j in range(mu.shape[1]):
        for b, band in enumerate(BAND_NAMES+['uncaptured']):
            MU_FREQUENCY_ROWS.append({'run_name':run.run_name,'beta_per_dim':beta_per_dim,'neuron_id':j,'band':band,
                                      'energy_abs':profile['energy'][j,b],'energy_fraction':profile['fraction'][j,b],
                                      'KL_j':profile['KL'][j],'var_mu_j':profile['variance'][j],'active_unit':bool(profile['active'][j])})
    im=ax.imshow(profile['fraction'],aspect='auto',cmap='viridis',vmin=0,vmax=1)
    ax.set_xticks(range(len(BAND_NAMES)+1),BAND_NAMES+['uncaptured'],rotation=45)
    ax.set_yticks(range(mu.shape[1]));ax.set_ylabel('Model-local latent dimension');ax.set_title(f'μ · β/d={beta_per_dim:g}')
fig.colorbar(im,ax=list(axes[0]),label='Fraction of centred μ energy',shrink=.7)
fig.suptitle(f'd={PLOT_D}, seed={PLOT_SEED} · same input graph; undefined cells = constant μ')
fig.savefig(OUT / 'latent_mu_frequency_heatmaps.png',dpi=170,bbox_inches='tight');display(fig);plt.close(fig)
MU_FREQUENCIES=pd.DataFrame(MU_FREQUENCY_ROWS)
MU_FREQUENCIES.to_parquet(OUT / 'latent_mu_frequency_profiles.parquet',index=False)
fig,ax=plt.subplots(figsize=(9,4))
for band,g in MU_FREQUENCIES.groupby('band'):
    curve=g.groupby('beta_per_dim').energy_abs.sum();ax.plot(curve.index,curve.values,marker='o',label=band)
ax.set_xscale('symlog',linthresh=5e-6);ax.set_xlabel('β/d');ax.set_ylabel('Absolute centred μ energy, summed over dimensions')
ax.set_title('Absolute band energy: scale changes remain visible');ax.legend(ncol=4,fontsize=9)
fig.tight_layout();fig.savefig(OUT / 'latent_mu_absolute_band_energy.png',dpi=170);display(fig);plt.close(fig)


## Welche Klassen hängen mit welchen Frequenzbändern zusammen?

Die vorherige Übersicht mittelt über Klassen. Hier zeigen wir **jede Klasse einzeln**: ihren Anteil der Labelenergie pro Band und die Anreicherung gegenüber derselben kohortenzentrierten Rauschreferenz.

Beispiel: Ein BRCA-LumA-Indikator vergleicht LumA mit den anderen bekannten BRCA-Subtypen, nicht mit sämtlichen übrigen Krebsarten. Geschlecht über alle Typen und innerhalb der gemischten Typen bleiben getrennt. Das ist ein Zusammenhang der Klassenlabels mit dem **Eingabegraphen**, noch kein Nachweis, dass ein VAE diese Information erhält oder eine Klasse anhand eines Bands zuverlässig klassifiziert.

**Mehr Energie** und **mehr Anreicherung** sind verschiedene Aussagen: B6 hat 680 Moden. Ein großer Energieanteil in B6 kann deshalb ohne überdurchschnittliche Konzentration entstehen. Klassen mit nur wenigen Testproben und deren Häufigkeiten werden ausdrücklich gezeigt. B0 bleibt separat; es ist beim normalisierten Laplacian trotz Sample-Zentrierung nicht zwingend energiefrei.


In [ ]:
CLASS_FREQUENCIES = pd.read_parquet(OUT / 'task_frequency_profiles.parquet')
CLASS_FREQUENCIES['enrichment'] = CLASS_FREQUENCIES.energy_fraction / CLASS_FREQUENCIES.noise_expected_fraction.where(CLASS_FREQUENCIES.noise_expected_fraction > 1e-24)
CLASS_SUMMARY_ROWS = []
for (task, label), g in CLASS_FREQUENCIES.groupby(['task','class']):
    informative = g[g.band.ne('B0')].dropna(subset=['energy_fraction','enrichment'])
    row = {'task':task, 'class':label, 'n_positive_test':int(g.n_positive.iloc[0]),
           'n_known_test_cohort':int(g.n_known_cohort.iloc[0]), 'small_test_class':bool(g.n_positive.iloc[0] < 20)}
    if len(informative):
        energy = informative.loc[informative.energy_fraction.idxmax()]
        enriched = informative.loc[informative.enrichment.idxmax()]
        row.update(max_energy_band=energy.band, max_energy_fraction=energy.energy_fraction,
                   most_enriched_band=enriched.band, max_enrichment=enriched.enrichment)
    else:
        row.update(max_energy_band=None,max_energy_fraction=np.nan,most_enriched_band=None,max_enrichment=np.nan)
    CLASS_SUMMARY_ROWS.append(row)
CLASS_FREQUENCY_SUMMARY = pd.DataFrame(CLASS_SUMMARY_ROWS)
CLASS_FREQUENCY_SUMMARY.to_parquet(OUT / 'class_frequency_summary.parquet',index=False)
CLASS_FREQUENCIES.to_parquet(OUT / 'class_frequency_profiles.parquet',index=False)
display(CLASS_FREQUENCY_SUMMARY)

for task, g in CLASS_FREQUENCIES.groupby('task'):
    energy = g.pivot(index='class',columns='band',values='energy_fraction').reindex(columns=BAND_NAMES)
    enriched = g.pivot(index='class',columns='band',values='enrichment').reindex(index=energy.index,columns=BAND_NAMES)
    counts = g.groupby('class').n_positive.first()
    rowlabels = [f'{label} (n={int(counts[label])})' for label in energy.index]
    fig, axes = plt.subplots(1,2,figsize=(14,max(3.6,.48*len(energy)+1.6)))
    cmap = plt.get_cmap('viridis').copy();cmap.set_bad('#EEEEEE')
    for ax,matrix,title in [(axes[0],energy,'Class label energy fraction'),
                            (axes[1],np.log2(enriched.clip(lower=1e-6)),'log₂ enrichment vs. matched noise')]:
        im=ax.imshow(matrix,aspect='auto',cmap=cmap,vmin=0 if ax is axes[0] else None,vmax=1 if ax is axes[0] else None)
        ax.set_yticks(range(len(matrix)),rowlabels);ax.set_xticks(range(len(BAND_NAMES)),BAND_NAMES)
        ax.set_xlabel('Disjoint frequency band · low → high');ax.set_title(title);ax.grid(False)
        fig.colorbar(im,ax=ax)
    fig.suptitle(f'{task} · class-specific signals on the fixed TCGA input graph',fontweight='bold')
    fig.tight_layout();fig.savefig(OUT / f'class_frequency_{task}.png',dpi=170,bbox_inches='tight');display(fig);plt.close(fig)


### Interpretation

`max_energy_band` beschreibt, wo der größte Energieanteil liegt. `most_enriched_band` beschreibt die stärkste Konzentration gegenüber kohortenpassendem Rauschen — **keine Signifikanzprüfung**, keine Kausalität und kein harter Frequenzbedarf. Einzelne Klassen können über mehrere Bänder verteilt sein; deshalb die vollständigen Heatmaps beachten, statt nur das stärkste Band zu betrachten.

Für eine Erhaltungsfrage müssen anschließend die Klassenprofile mit der Klassen-AUC des VAE und der Rekonstruierbarkeit der jeweiligen Bandunterräume zusammen betrachtet werden. Kleine Klassen erfordern zusätzliche Unsicherheitsprüfungen.
